# Resolver - entity resolution front of the Manager (person mode)

Replaces the naive `retrieve_contact` name-match stub with `resolve_meeting_context`
(RAG.md build order). The user says who they are meeting; this resolves the *person*
from a per-user store before any content RAG.

Pipeline position: `utterance -> resolve_meeting_context -> (resolved profile | fallback) -> run_manager -> ...`

Built so far: Steps 1-7 (store, clue extraction, typo-robust candidates, confidence,
routing, disclosure/selection, interaction-based resolution) and Steps 8-10 (write-back:
PrepSession persistence, create-from-cold-start, fact updates, cautious merge).
Steps 11-13 add the intent router (front door) plus recall and update flows.
`%run common.ipynb` for the shared bootstrap (model/agents/TODAY).

In [1]:
import os
# Run from backend/ so `%run common.ipynb` resolves regardless of the kernel's launch dir.
if not os.path.exists("common.ipynb"):
    for _p in ("backend", "extras/socialApp/backend"):
        if os.path.exists(f"{_p}/common.ipynb"):
            os.chdir(_p)
            break
%run common.ipynb

common ready. cwd=backend TODAY=2026-06-16
TalkingPoint, TopicTheme, PlannerOutput, BrainstormOutput, render_brainstorm defined.


In [2]:
# Step 1 - schemas (resolver-specific; not in common).
from typing import Literal

ResolveStatus = Literal["resolved", "ambiguous", "clarify", "not_found"]
TargetType = Literal["person", "event", "group", "unknown"]


class Fact(BaseModel):
    text: str
    observed_at: str
    source: str = "user"
    confidence: str = "high"


class Person(BaseModel):
    id: str
    name: str
    aliases: list[str] = Field(default_factory=list)
    company: str | None = None
    school: str | None = None
    role: str | None = None
    clubs: list[str] = Field(default_factory=list)
    tags: list[str] = Field(default_factory=list)
    profile_summary: str = ""
    facts: list[Fact] = Field(default_factory=list)


class QueryClues(BaseModel):
    names: list[str] = Field(default_factory=list, description="Person names or nicknames mentioned.")
    dates: list[str] = Field(default_factory=list, description="Concrete dates; resolve relative ones via today.")
    event_names: list[str] = Field(default_factory=list, description="Events/venues, e.g. 'Velocity'.")
    topics: list[str] = Field(default_factory=list, description="Topics/interests mentioned.")
    affiliations: list[str] = Field(default_factory=list, description="Companies, schools, clubs, teams.")
    course_codes: list[str] = Field(default_factory=list, description="Course codes, e.g. 'CS449'.")
    relationship_words: list[str] = Field(default_factory=list, description="Relationship hints, e.g. 'friend'.")


class Candidate(BaseModel):
    target_type: TargetType = "person"
    target_id: str
    display_name: str
    evidence: list[str] = Field(default_factory=list)
    score: float
    name_score: float = 0.0


class ResolutionResult(BaseModel):
    status: ResolveStatus
    target_type: TargetType = "person"
    selected_targets: list[Candidate] = Field(default_factory=list)
    candidates: list[Candidate] = Field(default_factory=list)
    extracted_clues: QueryClues
    resolution_summary: str = ""


class Interaction(BaseModel):
    id: str
    person_id: str
    date: str | None = None
    raw_note: str = ""
    summary: str = ""
    topics: list[str] = Field(default_factory=list)
    commitments: list[str] = Field(default_factory=list)


class PrepSession(BaseModel):
    id: str
    user_id: str
    target_ids: list[str] = Field(default_factory=list)
    user_query: str
    generated_card: dict | None = None
    timestamp: str


print("Person, QueryClues, Candidate, ResolutionResult, Interaction, Fact, PrepSession defined.")

Person, QueryClues, Candidate, ResolutionResult, Interaction, Fact, PrepSession defined.


In [3]:
# Step 1 - per-user store (in-memory stand-in for a per-user DB, keyed by user_id).
# Seeded from the old CONTACTS stand-ins + a deliberate two-Alex ambiguity.
PEOPLE: dict[str, list[Person]] = {
    "u1": [
        Person(id="p_sarah", name="Sarah", role="data scientist",
               tags=["rock climbing", "bouldering", "Japan", "sourdough", "Formula 1"],
               profile_summary="Close friend from university; data scientist into rock "
                               "climbing, recently back from Japan, baking sourdough, follows F1."),
        Person(id="p_david", name="David", role="head of engineering",
               tags=["marathon", "specialty coffee", "new father"],
               profile_summary="My manager; marathon runner, roasts his own coffee, new dad."),
        Person(id="p_mia", name="Mia", role="veterinarian",
               tags=["hiking", "dogs", "cello", "science fiction"],
               profile_summary="First date; vet who hikes with two dogs, plays cello, loves sci-fi."),
        Person(id="p_alex_chen", name="Alex Chen", aliases=["Alex"], clubs=["Velocity"],
               tags=["internships", "Waterloo co-op"],
               profile_summary="Met at the Velocity startup event; talked internships and co-op."),
        Person(id="p_alex_li", name="Alex Li", aliases=["Alex"], clubs=["CS449 group"],
               tags=["CS449"], profile_summary="Member of my CS449 group project."),
        Person(id="p_jordan", name="Jordan", tags=["board games"],
               profile_summary="Met at a board-game meetup; into strategy board games."),
    ],
}


INTERACTIONS: dict[str, list[Interaction]] = {
    "u1": [
        Interaction(id="i_jordan_1", person_id="p_jordan", date="2026-05-20",
                    summary="Talked about applying to ML grad programs.",
                    topics=["machine learning", "grad school"]),
    ],
}


def get_people(user_id: str) -> list[Person]:
    return PEOPLE.get(user_id, [])


def add_person(user_id: str, person: Person) -> None:
    PEOPLE.setdefault(user_id, []).append(person)


def get_person(user_id: str, target_id: str) -> Person | None:
    return next((p for p in get_people(user_id) if p.id == target_id), None)


def get_interactions(user_id: str) -> list[Interaction]:
    return INTERACTIONS.get(user_id, [])


print(f"store ready: u1 has {len(get_people('u1'))} people, "
      f"{len(get_interactions('u1'))} interactions.")

store ready: u1 has 6 people, 1 interactions.


In [4]:
# Step 3 - typo-robust name matching (inline, dependency-free).
def _norm(s: str) -> str:
    return s.lower().strip()


def damerau_levenshtein(a: str, b: str) -> int:
    """Optimal string alignment distance (adjacent transposition = 1 op)."""
    la, lb = len(a), len(b)
    d = [[0] * (lb + 1) for _ in range(la + 1)]
    for i in range(la + 1):
        d[i][0] = i
    for j in range(lb + 1):
        d[0][j] = j
    for i in range(1, la + 1):
        for j in range(1, lb + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            d[i][j] = min(d[i - 1][j] + 1, d[i][j - 1] + 1, d[i - 1][j - 1] + cost)
            if i > 1 and j > 1 and a[i - 1] == b[j - 2] and a[i - 2] == b[j - 1]:
                d[i][j] = min(d[i][j], d[i - 2][j - 2] + 1)
    return d[la][lb]


_SOUNDEX = {**dict.fromkeys("BFPV", "1"), **dict.fromkeys("CGJKQSXZ", "2"),
            **dict.fromkeys("DT", "3"), **dict.fromkeys("L", "4"),
            **dict.fromkeys("MN", "5"), **dict.fromkeys("R", "6")}


def soundex(name: str) -> str:
    name = "".join(c for c in name.upper() if c.isalpha())
    if not name:
        return ""
    digits = name[0]
    prev = _SOUNDEX.get(name[0], "")
    for ch in name[1:]:
        c = _SOUNDEX.get(ch, "")
        if c:
            if c != prev:
                digits += c
            prev = c
        elif ch not in "HW":  # vowels/Y reset, so same code across a vowel is kept
            prev = ""
        if len(digits) >= 4:
            break
    return (digits + "000")[:4]


def _name_tokens(p: Person) -> set[str]:
    toks: set[str] = set()
    for src in [p.name, *p.aliases]:
        for part in src.split():
            toks.add(_norm(part))
    return toks


def name_match_score(query_name: str, p: Person) -> float:
    """0..1. Exact token = 1.0; phonetic (soundex) hit = 0.9; else edit-distance scaled."""
    q = _norm(query_name)
    best = 0.0
    for tok in _name_tokens(p):
        if q == tok:
            return 1.0
        m = max(len(q), len(tok)) or 1
        s = 1 - damerau_levenshtein(q, tok) / m
        if soundex(q) == soundex(tok):
            s = max(s, 0.9)
        best = max(best, s)
    return best


def _term_hit(terms: list[str], fields: list[str]) -> bool:
    blob = " ".join(f for f in fields if f).lower()
    return any(_norm(t) in blob for t in terms)


print("name matching (damerau_levenshtein, soundex, name_match_score) defined.")

name matching (damerau_levenshtein, soundex, name_match_score) defined.


In [5]:
# Step 3 - candidate generation. Score ONLY against clue types present in the
# utterance; a missing clue type is not held against a candidate. Step 7 adds
# Interaction records as an extra evidence source (resolve by what was discussed).
NAME_FLOOR = 0.6  # below this, a name is not the same person


def _matching_interaction(terms: list[str], interactions: list[Interaction]):
    """First interaction whose topics / summary / raw_note contains a clue term."""
    for it in interactions:
        if _term_hit(terms, [*it.topics, it.summary, it.raw_note]):
            return it
    return None


def generate_candidates(clues: QueryClues, people: list[Person],
                        interactions: list[Interaction] | None = None) -> list[Candidate]:
    interactions = interactions or []
    present = [t for t in ("names", "affiliations", "event_names", "course_codes", "topics")
               if getattr(clues, t)]
    cands: list[Candidate] = []
    for p in people:
        p_inter = [it for it in interactions if it.person_id == p.id]
        inter_text = ([t for it in p_inter for t in it.topics]
                      + [it.summary for it in p_inter] + [it.raw_note for it in p_inter])
        fact_text = [f.text for f in p.facts]
        matched = 0.0
        name_hit = 0.0
        evidence: list[str] = []

        if clues.names:
            ns = max(name_match_score(n, p) for n in clues.names)
            if ns < NAME_FLOOR:
                continue  # a name was asked but this is not that person
            matched += ns
            name_hit = ns
            evidence.append(f"name~{ns:.2f}: {p.name}")

        aff_fields = [p.company or "", p.school or "", *p.clubs, *p.tags, *inter_text, *fact_text]
        for ctype in ("affiliations", "event_names", "course_codes"):
            terms = getattr(clues, ctype)
            if terms and _term_hit(terms, aff_fields):
                matched += 1.0
                evidence.append(f"{ctype}: {', '.join(terms)}")

        if clues.topics and _term_hit(clues.topics, [*p.tags, p.profile_summary, *inter_text, *fact_text]):
            matched += 1.0
            evidence.append(f"topics: {', '.join(clues.topics)}")

        # surface a matching past interaction as evidence (transparency for disclosure)
        clue_terms = clues.topics + clues.event_names + clues.course_codes + clues.affiliations
        hit = _matching_interaction(clue_terms, p_inter) if clue_terms else None
        if hit and hit.summary:
            evidence.append(f"via interaction: {hit.summary}")

        if not evidence:
            continue
        score = matched / len(present) if present else 0.0
        cands.append(Candidate(target_id=p.id, display_name=p.name, evidence=evidence,
                               score=round(score, 3), name_score=round(name_hit, 3)))
    cands.sort(key=lambda c: c.score, reverse=True)
    return cands


print("generate_candidates defined.")

generate_candidates defined.


In [6]:
# Step 4 - confidence rules: candidates -> ResolutionResult.
STRONG = 0.75  # build-time threshold; pinned by the demo cases below.
MARGIN = 0.20


def decide(clues: QueryClues, candidates: list[Candidate]) -> ResolutionResult:
    base = dict(extracted_clues=clues, candidates=candidates)
    if not candidates:
        return ResolutionResult(status="not_found", resolution_summary="No matching contact.", **base)
    top = candidates[0]
    if len(candidates) == 1:
        # A unique strong name should survive incidental or noisy clue extraction
        # like "coffee with Sarah" -> topics=["coffee"].
        if top.score >= STRONG or top.name_score >= STRONG:
            return ResolutionResult(status="resolved", selected_targets=[top],
                                    resolution_summary=f"Resolved to {top.display_name}.", **base)
        return ResolutionResult(status="clarify",
                                resolution_summary=f"Only a weak match ({top.display_name}).", **base)
    margin = top.score - candidates[1].score
    if top.score >= STRONG and margin >= MARGIN:
        return ResolutionResult(status="resolved", selected_targets=[top],
                                resolution_summary=f"Resolved to {top.display_name}.", **base)
    return ResolutionResult(status="ambiguous",
                            resolution_summary="Multiple close matches; ask the user.", **base)


def resolve_from_clues(clues: QueryClues, people: list[Person],
                       interactions: list[Interaction] | None = None) -> ResolutionResult:
    """Deterministic core (no LLM): clues + people (+ interactions) -> ResolutionResult."""
    return decide(clues, generate_candidates(clues, people, interactions))


print("decide + resolve_from_clues defined.")

decide + resolve_from_clues defined.


In [7]:
# Step 2 - LLM clue extraction (utterance -> QueryClues). One call, no tools.
CLUES_MODEL  = os.getenv("CLUES_MODEL",  "openai/gpt-4.1-mini")
CLUES_EFFORT = os.getenv("CLUES_EFFORT", "none")

CLUES_PROMPT = f"""\
You extract structured search clues from a short utterance about meeting or
recalling a person. You do not guess and you do not add anything not stated.

For reference, today's date is {TODAY}.

Fill QueryClues from ONLY what the utterance explicitly contains:
  names               person names or nicknames ("Alex", "Sarah")
  dates               concrete dates; convert relative ones ("tomorrow",
                      "last week") to ISO dates using today's date
  event_names         events or venues ("Velocity", "the career fair")
  topics              interests or identity clues about the person ("internships", "pottery")
  affiliations        companies, schools, clubs, teams ("Shopify", "the soccer club")
  course_codes        course codes ("CS449", "MATH239")
  relationship_words  relationship hints ("friend", "coworker", "my manager")

Important exclusions:
- Do NOT put meeting logistics or the occasion into topics: coffee, dinner,
  lunch, call, weekend plans, "meeting", "seeing", "grabbing coffee", etc.
  Only include those when the utterance says the person personally likes or is
  known for them, e.g. "David is into specialty coffee".
- Do NOT put verbs like meeting/seeing/grabbing into relationship_words.

Leave a field as an empty list when the utterance does not mention it. Never
invent a name, date, or affiliation that is not in the text.
"""

clues_agent = Agent(
    name="CluesAgent",
    instructions=CLUES_PROMPT,
    model=model(CLUES_MODEL),
    model_settings=model_settings(CLUES_EFFORT),
    output_type=QueryClues,
)


async def extract_clues(utterance: str, now: datetime.date | None = None) -> QueryClues:
    now = now or datetime.date.today()
    msg = f"Utterance: {utterance}\nToday: {now.isoformat()}"
    result = await Runner.run(clues_agent, msg, max_turns=2)
    return result.final_output


print("clues_agent + extract_clues defined.")

clues_agent + extract_clues defined.


In [8]:
# Steps 2+3+4 - the full entry point that replaces retrieve_contact.
async def resolve_meeting_context(
    user_id: str, utterance: str, now: datetime.date | None = None
) -> ResolutionResult:
    """utterance -> extract clues (LLM) -> candidates over the user's people -> decide."""
    now = now or datetime.date.today()
    clues = await extract_clues(utterance, now)
    return resolve_from_clues(clues, get_people(user_id), get_interactions(user_id))


def show_resolution(r: ResolutionResult) -> None:
    print(f"status: {r.status}  ({r.resolution_summary})")
    print(f"clues:  {r.extracted_clues.model_dump(exclude_defaults=True)}")
    for c in r.candidates:
        mark = "*" if c in r.selected_targets else " "
        print(f"  {mark} {c.display_name}  score={c.score}  {c.evidence}")


print("resolve_meeting_context + show_resolution defined.")

resolve_meeting_context + show_resolution defined.


In [9]:
# Step 5 - route a ResolutionResult into the next pipeline action (pure; no LLM).
class RouteDecision(BaseModel):
    action: Literal["prep", "disambiguate", "fallback"]
    contact_info: str | None = None   # profile_summary when action == "prep"
    message: str | None = None        # choice list shown to the user when disambiguate


def render_contact_info(person: Person | None, candidate: Candidate | None = None) -> str | None:
    """Profile context passed to Manager after resolution.
    Includes structured person fields and non-name evidence, not just the summary.
    """
    if person is None:
        return None
    lines = [person.profile_summary]
    if person.role:
        lines.append(f"Role: {person.role}")
    if person.company:
        lines.append(f"Company: {person.company}")
    if person.school:
        lines.append(f"School: {person.school}")
    if person.clubs:
        lines.append(f"Clubs/groups: {', '.join(person.clubs)}")
    if person.tags:
        lines.append(f"Tags/topics: {', '.join(person.tags)}")
    if person.facts:
        lines.append("Confirmed facts: " + "; ".join(f.text for f in person.facts))
    if candidate:
        useful_evidence = [ev for ev in candidate.evidence if not ev.startswith("name~")]
        if useful_evidence:
            lines.append("Resolution evidence: " + "; ".join(useful_evidence))
    return "\n".join(line for line in lines if line)


def route_resolution(resolution: ResolutionResult, user_id: str) -> RouteDecision:
    """resolved -> prep with the person's profile; ambiguous -> disambiguate (never
    fall through to a generic card); not_found / clarify -> utterance-only fallback."""
    if resolution.status == "resolved" and resolution.selected_targets:
        candidate = resolution.selected_targets[0]
        person = get_person(user_id, candidate.target_id)
        return RouteDecision(action="prep",
                             contact_info=render_contact_info(person, candidate))
    if resolution.status == "ambiguous":
        lines = ["Did you mean:"]
        for n, c in enumerate(resolution.candidates, 1):
            lines.append(f"  {n}. {c.display_name} - {', '.join(c.evidence)}")
        return RouteDecision(action="disambiguate", message="\n".join(lines))
    return RouteDecision(action="fallback")  # not_found / clarify


print("RouteDecision + route_resolution defined.")

RouteDecision + route_resolution defined.


In [10]:
# Step 6 - progressive disclosure: turn an ambiguous result into a resolved one
# once the user picks a candidate (1-based index, or a target_id).
def select_candidate(resolution: ResolutionResult, choice: int | str) -> ResolutionResult:
    cands = resolution.candidates
    chosen = None
    if isinstance(choice, str) and choice.isdigit():
        choice = int(choice)   # accept "2" as an index, not a target_id
    if isinstance(choice, int):
        if 1 <= choice <= len(cands):
            chosen = cands[choice - 1]
    else:
        chosen = next((c for c in cands if c.target_id == choice), None)
    if chosen is None:
        return resolution  # invalid pick -> unchanged (still ambiguous)
    return ResolutionResult(
        status="resolved", selected_targets=[chosen], candidates=cands,
        extracted_clues=resolution.extracted_clues,
        resolution_summary=f"Resolved to {chosen.display_name} (user choice).",
    )


print("select_candidate defined.")

select_candidate defined.


In [11]:
# Steps 8-10 - write-back: persist prep sessions, create people from cold start,
# update facts, and merge duplicates. Every mutation is explicit (propose vs apply),
# so the caller/UI confirms before anything is written. In-memory store for now.
import uuid


def _new_id(prefix: str) -> str:
    return f"{prefix}_{uuid.uuid4().hex[:8]}"


# Step 8 - PrepSession persistence
PREP_SESSIONS: dict[str, list[PrepSession]] = {}


def save_prep_session(user_id: str, user_query: str, target_ids: list[str],
                      generated_card: dict | None = None) -> PrepSession:
    ps = PrepSession(id=_new_id("ps"), user_id=user_id, user_query=user_query,
                     target_ids=target_ids, generated_card=generated_card,
                     timestamp=datetime.datetime.now().isoformat(timespec="seconds"))
    PREP_SESSIONS.setdefault(user_id, []).append(ps)
    return ps


def get_prep_sessions(user_id: str) -> list[PrepSession]:
    return PREP_SESSIONS.get(user_id, [])


# Step 9 - create a new Person from a not_found prep (seeds out of cold start).
# `mo` is a ManagerOutput (duck-typed: .subjects[].name, .overall_context) so this
# notebook does not need the Manager imported.
def propose_new_person(resolution: ResolutionResult, mo) -> Person | None:
    """Unsaved Person from a not_found utterance + Manager output. None if no name."""
    names = resolution.extracted_clues.names
    if not names:
        return None
    return Person(id=_new_id("p"), name=names[0],
                  tags=[s.name for s in mo.subjects],
                  profile_summary=mo.overall_context)


# Step 10 - cautious merge: a new mention that matches an existing person.
def find_merge_candidates(user_id: str, name: str, threshold: float = STRONG) -> list[Candidate]:
    """Existing people whose name strongly matches `name` (the duplicate check)."""
    cands = generate_candidates(QueryClues(names=[name]), get_people(user_id))
    return [c for c in cands if c.name_score >= threshold]


def save_new_person(user_id: str, person: Person):
    """Create UNLESS a strong existing match exists -> then propose a merge instead of
    writing a duplicate. Returns ('created', Person) or ('merge_suggested', [Candidate])."""
    dupes = find_merge_candidates(user_id, person.name)
    if dupes:
        return "merge_suggested", dupes
    add_person(user_id, person)
    return "created", person


def apply_merge(user_id: str, keep_id: str, incoming: Person) -> Person | None:
    """Fold `incoming` (assumed not yet persisted) into existing person `keep_id`."""
    keep = get_person(user_id, keep_id)
    if keep is None:
        return None

    def _union(a, b):
        return list(dict.fromkeys([*a, *b]))

    if incoming.name and incoming.name != keep.name:
        keep.aliases = _union(keep.aliases, [incoming.name])
    keep.aliases = _union(keep.aliases, incoming.aliases)
    keep.clubs = _union(keep.clubs, incoming.clubs)
    keep.tags = _union(keep.tags, incoming.tags)
    keep.facts = [*keep.facts, *incoming.facts]
    if incoming.profile_summary and incoming.profile_summary not in keep.profile_summary:
        keep.profile_summary = (keep.profile_summary + " " + incoming.profile_summary).strip()
    return keep


# Step 10 - confirmed fact updates (time-stamped observations; never overwrite).
def propose_fact(text: str, source: str = "user", confidence: str = "high") -> Fact:
    """Unsaved Fact. Caller confirms, then calls add_fact to commit."""
    return Fact(text=text, observed_at=datetime.datetime.now().isoformat(timespec="seconds"),
                source=source, confidence=confidence)


def add_fact(user_id: str, person_id: str, fact: Fact) -> Person | None:
    p = get_person(user_id, person_id)
    if p is None:
        return None
    p.facts.append(fact)
    return p


print("write-back defined: save_prep_session / get_prep_sessions / propose_new_person / "
      "save_new_person / find_merge_candidates / apply_merge / propose_fact / add_fact")

write-back defined: save_prep_session / get_prep_sessions / propose_new_person / save_new_person / find_merge_candidates / apply_merge / propose_fact / add_fact


In [12]:
# Step 11 - intent router (front door). One LLM call; default to 'prep' when unsure.
Intent = Literal["prep", "recall", "update"]
INTENT_MODEL  = os.getenv("INTENT_MODEL",  "openai/gpt-4.1-mini")
INTENT_EFFORT = os.getenv("INTENT_EFFORT", "none")


class IntentResult(BaseModel):
    intent: Intent = Field(
        description="prep = preparing to meet someone; recall = asking ABOUT a person from "
        "memory; update = asserting a fact to store about a person.")


INTENT_PROMPT = """\
Classify a short utterance about another person into ONE intent:
  prep    - the user is preparing to meet or see someone
            ("I'm meeting Alex", "coffee with Sarah this weekend").
  recall  - the user is asking ABOUT a person from memory
            ("who was that person from Velocity?", "what's her name again", "tell me about Alex").
  update  - the user is recording a fact about a person
            ("update Weilong's profile", "Alex got an internship at Shopify", "she's into pottery now").
When the utterance is ambiguous, choose prep. Output only the intent.
"""

intent_agent = Agent(name="IntentAgent", instructions=INTENT_PROMPT,
                     model=model(INTENT_MODEL), model_settings=model_settings(INTENT_EFFORT),
                     output_type=IntentResult)


async def classify_intent(utterance: str) -> Intent:
    r = await Runner.run(intent_agent, utterance, max_turns=2)
    return r.final_output.intent


print("intent_agent + classify_intent defined.")

intent_agent + classify_intent defined.


In [13]:
# Steps 12-13 - recall (read-only answer) and update (write a fact) flows.
# Deterministic cores (render/apply) are testable offline; the LLM extractor is live.

# Step 12 - recall
class RecallResult(BaseModel):
    action: Literal["answer", "disambiguate", "not_found"]
    message: str
    person_id: str | None = None


def render_recall_answer(person: Person, interactions: list[Interaction] | None = None) -> str:
    """A short, grounded answer read straight from the record (no LLM, no research)."""
    lines = [person.name + (f" - {person.role}" if person.role else "")]
    if person.tags:
        lines.append("Interests: " + ", ".join(person.tags))
    if person.facts:
        lines.append("Recent: " + "; ".join(f.text for f in person.facts))
    its = [it for it in (interactions or []) if it.person_id == person.id]
    if its:
        lines.append("You last noted: " + its[-1].summary)
    if person.profile_summary:
        lines.append(person.profile_summary)
    return "\n".join(lines)


def apply_recall(resolution: ResolutionResult, user_id: str) -> RecallResult:
    if resolution.status == "resolved" and resolution.selected_targets:
        p = get_person(user_id, resolution.selected_targets[0].target_id)
        return RecallResult(action="answer", person_id=p.id,
                            message=render_recall_answer(p, get_interactions(user_id)))
    if resolution.status == "ambiguous":
        return RecallResult(action="disambiguate",
                            message=route_resolution(resolution, user_id).message)
    return RecallResult(action="not_found", message="I don't have anyone like that on file.")


async def run_recall(user_id: str, utterance: str) -> RecallResult:
    return apply_recall(await resolve_meeting_context(user_id, utterance), user_id)


# Step 13 - update
UPDATE_MODEL  = os.getenv("UPDATE_MODEL",  "openai/gpt-4.1-mini")
UPDATE_EFFORT = os.getenv("UPDATE_EFFORT", "none")


class UpdateExtraction(BaseModel):
    target_name: str = Field(description="Who the fact is about, e.g. 'Weilong'.")
    fact: str = Field(description="The asserted fact as a short statement, e.g. 'is into climbing'.")


UPDATE_PROMPT = """\
The user wants to record a fact about a person. Extract:
  target_name - who the fact is about (a name).
  fact        - the asserted fact, restated as a short statement about them.
Example: "update Weilong's profile, he is really into workout lately"
  -> target_name="Weilong", fact="is really into working out lately".
Only restate what the user said; do not invent facts.
"""

update_extractor_agent = Agent(name="UpdateExtractor", instructions=UPDATE_PROMPT,
                               model=model(UPDATE_MODEL), model_settings=model_settings(UPDATE_EFFORT),
                               output_type=UpdateExtraction)


async def extract_update(utterance: str) -> UpdateExtraction:
    r = await Runner.run(update_extractor_agent, utterance, max_turns=2)
    return r.final_output


class UpdateResult(BaseModel):
    action: Literal["confirm_update", "updated", "confirm_create", "created", "disambiguate"]
    message: str
    person_id: str | None = None


def apply_update(resolution: ResolutionResult, fact_text: str, user_id: str,
                 confirm: bool = False) -> UpdateResult:
    """Write a confirmed fact. confirm=False proposes (no write); confirm=True commits.
    resolved -> append the fact; not_found/clarify -> create the contact, then append."""
    if resolution.status == "ambiguous":
        return UpdateResult(action="disambiguate",
                            message=route_resolution(resolution, user_id).message)
    fact = propose_fact(fact_text)
    if resolution.status == "resolved" and resolution.selected_targets:
        p = get_person(user_id, resolution.selected_targets[0].target_id)
        if not confirm:
            return UpdateResult(action="confirm_update", person_id=p.id,
                                message=f'Add to {p.name}: "{fact_text}"?')
        add_fact(user_id, p.id, fact)
        return UpdateResult(action="updated", person_id=p.id, message=f"Updated {p.name}.")
    names = resolution.extracted_clues.names
    if not names:
        return UpdateResult(action="disambiguate", message="Who is this about?")
    if not confirm:
        return UpdateResult(action="confirm_create",
                            message=f'I have no "{names[0]}". Create them and add "{fact_text}"?')
    person = Person(id=_new_id("p"), name=names[0])
    add_person(user_id, person)
    add_fact(user_id, person.id, fact)
    return UpdateResult(action="created", person_id=person.id,
                        message=f"Created {names[0]} and added the fact.")


async def run_update(user_id: str, utterance: str, confirm: bool = False) -> UpdateResult:
    ext = await extract_update(utterance)
    resolution = resolve_from_clues(QueryClues(names=[ext.target_name]),
                                    get_people(user_id), get_interactions(user_id))
    return apply_update(resolution, ext.fact, user_id, confirm=confirm)


print("recall (render_recall_answer/apply_recall/run_recall) + "
      "update (extract_update/apply_update/run_update) defined.")

recall (render_recall_answer/apply_recall/run_recall) + update (extract_update/apply_update/run_update) defined.


In [14]:
# Demo - deterministic tests for Steps 1/3/4 (no network). Gated so `%run` stays quiet.
RUN_DEMO = globals().get("RUN_DEMO", True)
if RUN_DEMO:
    S  = next(p for p in get_people("u1") if p.id == "p_sarah")
    D  = next(p for p in get_people("u1") if p.id == "p_david")
    AC = next(p for p in get_people("u1") if p.id == "p_alex_chen")
    AL = next(p for p in get_people("u1") if p.id == "p_alex_li")
    fails = 0

    def _check(label, got, want):
        global fails
        ok = got == want
        fails += 0 if ok else 1
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}: {got!r} (want {want!r})")

    print("Step 1 - store:")
    _check("u1 count", len(get_people("u1")), 6)
    _check("unknown user", get_people("nobody"), [])

    print("Step 3+4 - resolution (hand-fed clues; LLM extraction tested in the live cell):")
    _check("T1 unique 'Alex'",        resolve_from_clues(QueryClues(names=["Alex"]),  [AC, S, D]).status,  "resolved")
    _check("T2 two Alexes",           resolve_from_clues(QueryClues(names=["Alex"]),  [AC, AL, S]).status, "ambiguous")
    _check("T3 unknown 'Zoltan'",     resolve_from_clues(QueryClues(names=["Zoltan"]),[AC, AL, S, D]).status, "not_found")
    _check("T4 typo 'Alxe' unique",   resolve_from_clues(QueryClues(names=["Alxe"]),  [AC, S]).status,     "resolved")
    _check("T5 typo 'Alxe' 2 Alexes", resolve_from_clues(QueryClues(names=["Alxe"]),  [AC, AL]).status,    "ambiguous")
    r6 = resolve_from_clues(QueryClues(names=["Alex"], course_codes=["CS449"]), [AC, AL])
    _check("T6 'Alex'+CS449 status",  r6.status, "resolved")
    _check("T6 selected = Alex Li",   r6.selected_targets[0].target_id if r6.selected_targets else None, "p_alex_li")
    weak = [Candidate(target_id="p_x", display_name="X", score=0.65, evidence=["name~0.65"])]
    _check("T7 weak single -> clarify", decide(QueryClues(names=["X"]), weak).status, "clarify")
    _check("T8 unique name + incidental topic", resolve_from_clues(QueryClues(names=["Sarah"], topics=["coffee"]), [S, AC, AL]).status, "resolved")

    print("Step 5 - routing:")
    rd_res = route_resolution(resolve_from_clues(QueryClues(names=["Sarah"]), get_people("u1")), "u1")
    _check("R1 resolved -> prep", rd_res.action, "prep")
    _check("R1 contact = Sarah profile", bool(rd_res.contact_info and rd_res.contact_info.startswith("Close friend")), True)
    rd_amb = route_resolution(resolve_from_clues(QueryClues(names=["Alex"]), get_people("u1")), "u1")
    _check("R2 ambiguous -> disambiguate", rd_amb.action, "disambiguate")
    _check("R2 lists both Alexes", ("Alex Chen" in rd_amb.message and "Alex Li" in rd_amb.message), True)
    _check("R3 not_found -> fallback", route_resolution(resolve_from_clues(QueryClues(names=["Zoltan"]), get_people("u1")), "u1").action, "fallback")
    rd_clar = route_resolution(decide(QueryClues(names=["X"]), [Candidate(target_id="p_x", display_name="X", score=0.65, name_score=0.65, evidence=["name~0.65"])]), "u1")
    _check("R4 clarify -> fallback", rd_clar.action, "fallback")

    print("Step 6 - disclosure (pick from ambiguous):")
    amb = resolve_from_clues(QueryClues(names=["Alex"]), get_people("u1"))
    _check("S6a pick #2 -> resolved", select_candidate(amb, 2).status, "resolved")
    _check("S6b pick by id -> that person", select_candidate(amb, "p_alex_li").selected_targets[0].target_id, "p_alex_li")
    _check("S6c invalid pick -> still ambiguous", select_candidate(amb, 99).status, "ambiguous")
    _check("S6d picked -> routes to prep", route_resolution(select_candidate(amb, 1), "u1").action, "prep")
    _check("S6e str-index pick -> resolved", select_candidate(amb, "2").status, "resolved")

    print("Step 7 - interaction-based resolution:")
    r7 = resolve_from_clues(QueryClues(topics=["machine learning"]), get_people("u1"), get_interactions("u1"))
    _check("S7a topic via interaction -> resolved", r7.status, "resolved")
    _check("S7a resolved to Jordan", r7.selected_targets[0].target_id if r7.selected_targets else None, "p_jordan")
    _check("S7b no interactions -> not_found", resolve_from_clues(QueryClues(topics=["machine learning"]), get_people("u1"), []).status, "not_found")
    _check("S7c interaction shown as evidence", any("via interaction" in e for c in r7.candidates for e in c.evidence), True)

    print("Aux - typo internals:")
    _check("soundex(Alxe)==soundex(Alex)", soundex("Alxe") == soundex("Alex"), True)
    _check("DL(alxe,alex)", damerau_levenshtein("alxe", "alex"), 1)

    print("Steps 8-10 - write-back (isolated user u_test):")
    from types import SimpleNamespace
    U = "u_test"
    PEOPLE[U] = []
    PREP_SESSIONS.pop(U, None)
    nf = ResolutionResult(status="not_found", extracted_clues=QueryClues(names=["Weilong"]))
    fake_mo = SimpleNamespace(
        subjects=[SimpleNamespace(name="workout"), SimpleNamespace(name="grad school")],
        overall_context="Meeting Weilong, a friend getting into fitness.")
    proposed = propose_new_person(nf, fake_mo)
    _check("S9a propose builds a Person", proposed.name if proposed else None, "Weilong")
    _check("S9b not created until saved", len(get_people(U)), 0)
    kind, _obj = save_new_person(U, proposed)
    _check("S9c saved -> created", kind, "created")
    _check("S9d store grew", len(get_people(U)), 1)
    _check("S9e Weilong now resolves", resolve_from_clues(QueryClues(names=["Weilong"]), get_people(U)).status, "resolved")
    _check("S9f no name -> no proposal", propose_new_person(ResolutionResult(status="not_found", extracted_clues=QueryClues()), fake_mo), None)
    save_prep_session(U, "dinner with Weilong", [proposed.id], {"overall_context": "x"})
    _check("S8a session persisted", len(get_prep_sessions(U)), 1)
    _check("S8b session links person", get_prep_sessions(U)[0].target_ids, [proposed.id])
    kind2, _dupes = save_new_person(U, Person(id=_new_id("p"), name="Weilong", tags=["climbing"]))
    _check("S10a duplicate -> merge_suggested", kind2, "merge_suggested")
    _check("S10b no dup row added", len(get_people(U)), 1)
    merged = apply_merge(U, proposed.id, Person(id="tmp", name="Weilong", tags=["climbing"]))
    _check("S10c merge folds tags", "climbing" in merged.tags, True)
    f = propose_fact("Weilong is really into workout lately")
    _check("S10d propose doesn't write", len(get_person(U, proposed.id).facts), 0)
    add_fact(U, proposed.id, f)
    _check("S10e fact appended", get_person(U, proposed.id).facts[0].text.startswith("Weilong"), True)
    _check("S10f fact searchable", resolve_from_clues(QueryClues(topics=["workout"]), get_people(U)).selected_targets[0].target_id, proposed.id)
    _check("S10g fact rendered", "workout" in render_contact_info(get_person(U, proposed.id)), True)

    print("Step 12 - recall (deterministic answer rendering):")
    rr = apply_recall(resolve_from_clues(QueryClues(names=["Sarah"]), get_people("u1")), "u1")
    _check("S12a resolved -> answer", rr.action, "answer")
    _check("S12b answer grounded", "Sarah" in rr.message and "rock climbing" in rr.message, True)
    _check("S12c ambiguous -> disambiguate", apply_recall(resolve_from_clues(QueryClues(names=["Alex"]), get_people("u1")), "u1").action, "disambiguate")
    _check("S12d not_found -> not_found", apply_recall(resolve_from_clues(QueryClues(names=["Zoltan"]), get_people("u1")), "u1").action, "not_found")
    Jp = next(p for p in get_people("u1") if p.id == "p_jordan")
    _check("S12e render uses interaction", "You last noted" in render_recall_answer(Jp, get_interactions("u1")), True)

    print("Step 13 - update (deterministic write given an extracted fact):")
    UU = "u_upd"
    PEOPLE[UU] = [Person(id="p_w", name="Weilong", profile_summary="A friend.")]
    res_r = resolve_from_clues(QueryClues(names=["Weilong"]), get_people(UU))
    _check("S13a resolved+propose -> confirm_update", apply_update(res_r, "is into climbing", UU, confirm=False).action, "confirm_update")
    _check("S13b no write before confirm", len(get_person(UU, "p_w").facts), 0)
    _check("S13c confirm -> updated", apply_update(res_r, "is into climbing", UU, confirm=True).action, "updated")
    _check("S13d fact written", get_person(UU, "p_w").facts[0].text, "is into climbing")
    res_nf = resolve_from_clues(QueryClues(names=["Brandnew"]), get_people(UU))
    _check("S13e not_found+propose -> confirm_create", apply_update(res_nf, "likes chess", UU, confirm=False).action, "confirm_create")
    _check("S13f no person before confirm", len(get_people(UU)), 1)
    out13 = apply_update(res_nf, "likes chess", UU, confirm=True)
    _check("S13g confirm -> created", out13.action, "created")
    _check("S13h new person + fact", get_person(UU, out13.person_id).facts[0].text, "likes chess")
    _check("S13i ambiguous update -> disambiguate", apply_update(resolve_from_clues(QueryClues(names=["Alex"]), get_people("u1")), "x", "u1", confirm=True).action, "disambiguate")

    print(f"\n{'ALL PASS' if fails == 0 else str(fails) + ' FAILURE(S)'}")

Step 1 - store:
  [PASS] u1 count: 6 (want 6)
  [PASS] unknown user: [] (want [])
Step 3+4 - resolution (hand-fed clues; LLM extraction tested in the live cell):
  [PASS] T1 unique 'Alex': 'resolved' (want 'resolved')
  [PASS] T2 two Alexes: 'ambiguous' (want 'ambiguous')
  [PASS] T3 unknown 'Zoltan': 'not_found' (want 'not_found')
  [PASS] T4 typo 'Alxe' unique: 'resolved' (want 'resolved')
  [PASS] T5 typo 'Alxe' 2 Alexes: 'ambiguous' (want 'ambiguous')
  [PASS] T6 'Alex'+CS449 status: 'resolved' (want 'resolved')
  [PASS] T6 selected = Alex Li: 'p_alex_li' (want 'p_alex_li')
  [PASS] T7 weak single -> clarify: 'clarify' (want 'clarify')
  [PASS] T8 unique name + incidental topic: 'resolved' (want 'resolved')
Step 5 - routing:
  [PASS] R1 resolved -> prep: 'prep' (want 'prep')
  [PASS] R1 contact = Sarah profile: True (want True)
  [PASS] R2 ambiguous -> disambiguate: 'disambiguate' (want 'disambiguate')
  [PASS] R2 lists both Alexes: True (want True)
  [PASS] R3 not_found -> fallbac

In [15]:
# Live demo - exercises the LLM clue extractor + full resolve_meeting_context.
# OFF by default (needs OPENROUTER network). Set RUN_LIVE = True to run.
RUN_LIVE = globals().get("RUN_LIVE", False)
if RUN_LIVE:
    for utt in [
        "I am meeting Alex tomorrow",
        "I am meeting Alxe from my CS449 group",
        "coffee with Sarah this weekend",
        "meeting Zoltan next week",
    ]:
        print("=" * 64); print(utt)
        show_resolution(await resolve_meeting_context("u1", utt))
        print()